
# PRAM Lab 01 — Reduction and CREW Prefix Sum

This version uses **C++ for the algorithmic part** and **Python only for visualization**.

The prefix-sum algorithm in this notebook is the **CREW block-doubling prefix sum**:

- at each round, blocks double in size;
- in each block, the **last prefix of the left half** is read concurrently by all processors in the right half;
- therefore the model is **CREW**;
- the work is **Θ(n log n)** because every round performs **n/2 additions**.


In [8]:

%%bash
# Compile the C++ backend used by the notebook.
set -e
cat > pram_lab.cpp <<'CPP'
#include <algorithm>
#include <cstdlib>
#include <iostream>
#include <map>
#include <sstream>
#include <string>
#include <vector>
using namespace std;

struct Op {
    int p=-1, left=-1, right=-1, out=-1;
    int block_start=-1, half=-1;
    long long before_left=0, before_right=0, result=0;
    string kind;
};

static string q(const string& s){
    string out="\"";
    for(char c: s){ if(c=='\"' || c=='\\') out += '\\'; out += c; }
    out += '"'; return out;
}
static void print_vec(const vector<long long>& v){
    cout << '[';
    for(size_t i=0;i<v.size();++i){ if(i) cout << ','; cout << v[i]; }
    cout << ']';
}
static void print_ops(const vector<Op>& ops){
    cout << '[';
    for(size_t i=0;i<ops.size();++i){
        if(i) cout << ',';
        const auto& o=ops[i];
        cout << "{\"p\":"<<o.p
             <<",\"kind\":"<<q(o.kind)
             <<",\"left\":"<<o.left
             <<",\"right\":"<<o.right
             <<",\"out\":"<<o.out
             <<",\"block_start\":"<<o.block_start
             <<",\"half\":"<<o.half
             <<",\"before_left\":"<<o.before_left
             <<",\"before_right\":"<<o.before_right
             <<",\"result\":"<<o.result << '}';
    }
    cout << ']';
}
static vector<long long> parse_values(int argc, char** argv, int start){
    vector<long long> v;
    for(int i=start;i<argc;++i) v.push_back(stoll(argv[i]));
    return v;
}
static bool power_of_two(size_t n){ return n && ((n & (n-1))==0); }

static void reduction(const vector<long long>& input){
    vector<vector<long long>> states;
    vector<vector<Op>> allops;
    states.push_back(input); allops.push_back({});
    vector<long long> cur=input;
    while(cur.size()>1){
        vector<long long> nxt(cur.size()/2);
        vector<Op> ops;
        for(size_t p=0;p<nxt.size();++p){
            int l=2*(int)p, r=l+1;
            nxt[p]=cur[l]+cur[r];
            Op o; o.p=(int)p; o.kind="add"; o.left=l; o.right=r; o.out=(int)p;
            o.before_left=cur[l]; o.before_right=cur[r]; o.result=nxt[p];
            ops.push_back(o);
        }
        states.push_back(nxt); allops.push_back(ops); cur=nxt;
    }
    cout << "{\"mode\":\"reduction\",\"states\":[";
    for(size_t r=0;r<states.size();++r){
        if(r) cout << ',';
        cout << "{\"round\":"<<r<<",\"values\":"; print_vec(states[r]);
        cout << ",\"ops\":"; print_ops(allops[r]); cout << '}';
    }
    cout << "]}";
}

static void scan_block(const vector<long long>& input){
    if(!power_of_two(input.size())){ cerr << "n must be a power of two\n"; exit(2); }
    struct State {
        int round, half, block_size, additions;
        string note;
        vector<long long> before, values;
        vector<Op> ops;
    };
    vector<State> states;
    vector<long long> src=input;
    states.push_back({0,0,1,0,"Input: each cell contains one element.",src,src,{}});
    int round=1;
    long long total_additions=0;
    for(int half=1; half<(int)src.size(); half*=2, ++round){
        int block_size = 2*half;
        vector<long long> dst=src;
        vector<Op> ops;
        for(int block_start=0; block_start<(int)src.size(); block_start += block_size){
            int pivot = block_start + half - 1;
            long long carry = src[pivot];
            for(int j=block_start + half; j<block_start + block_size && j<(int)src.size(); ++j){
                dst[j] = src[j] + carry;
                Op o; o.p=j; o.kind="block-add"; o.left=pivot; o.right=j; o.out=j;
                o.block_start=block_start; o.half=half;
                o.before_left=carry; o.before_right=src[j]; o.result=dst[j];
                ops.push_back(o);
            }
        }
        total_additions += (long long)ops.size();
        ostringstream note;
        note << "Round " << round << ": block size = " << block_size
             << ". In each block, the last prefix of the left half is read concurrently by all processors in the right half.";
        states.push_back({round,half,block_size,(int)ops.size(),note.str(),src,dst,ops});
        src.swap(dst);
    }
    cout << "{\"mode\":\"scan-block\",\"result\":"; print_vec(src);
    cout << ",\"total_additions\":"<<total_additions<<",\"states\":[";
    for(size_t k=0;k<states.size();++k){
        if(k) cout << ',';
        auto &s=states[k];
        cout << "{\"round\":"<<s.round<<",\"half\":"<<s.half
             <<",\"block_size\":"<<s.block_size
             <<",\"additions\":"<<s.additions
             <<",\"note\":"<<q(s.note)
             << ",\"before\":"; print_vec(s.before);
        cout << ",\"values\":"; print_vec(s.values);
        cout << ",\"ops\":"; print_ops(s.ops); cout << '}';
    }
    cout << "]}";
}

static void legality(const string& scenario, int p){
    map<string, vector<int>> reads, writes;
    if(scenario=="reduction"){
        for(int proc=0; proc<p; ++proc){
            reads["A["+to_string(2*proc)+"]"].push_back(proc);
            reads["A["+to_string(2*proc+1)+"]"].push_back(proc);
            writes["B["+to_string(proc)+"]"].push_back(proc);
        }
    } else if(scenario=="scan-block"){
        int n = p;
        int half = 2;
        int block_size = 2*half;
        for(int block_start=0; block_start<n; block_start += block_size){
            int pivot = block_start + half - 1;
            for(int j=block_start + half; j<block_start + block_size && j<n; ++j){
                reads["X["+to_string(pivot)+"]"].push_back(j);
                reads["X["+to_string(j)+"]"].push_back(j);
                writes["Y["+to_string(j)+"]"].push_back(j);
            }
        }
        for(int j=0;j<n;++j){
            if(!writes.count("Y["+to_string(j)+"]")) writes["Y["+to_string(j)+"]"].push_back(j);
        }
    } else { cerr << "unknown scenario\n"; exit(2); }
    bool read_conflict=false, write_conflict=false;
    for(auto& [a,ps]:reads) if(ps.size()>1) read_conflict=true;
    for(auto& [a,ps]:writes) if(ps.size()>1) write_conflict=true;
    cout << "{\"scenario\":"<<q(scenario)
         <<",\"read_conflict\":"<<(read_conflict?"true":"false")
         <<",\"write_conflict\":"<<(write_conflict?"true":"false")
         <<",\"EREW\":"<<((!read_conflict&&!write_conflict)?"true":"false")
         <<",\"CREW\":"<<((!write_conflict)?"true":"false")
         <<",\"CRCW\":true}";
}

int main(int argc, char** argv){
    if(argc<2){ cerr << "usage: pram_lab MODE ...\n"; return 2; }
    string mode=argv[1];
    if(mode=="reduction"){
        auto v=parse_values(argc,argv,2);
        if(!power_of_two(v.size())){cerr<<"reduction n must be power of two\n"; return 2;}
        reduction(v);
    } else if(mode=="scan"){
        auto v=parse_values(argc,argv,2); scan_block(v);
    } else if(mode=="legality"){
        if(argc!=4){cerr<<"usage legality scenario p\n"; return 2;}
        legality(argv[2],stoi(argv[3]));
    } else { cerr << "unknown mode\n"; return 2; }
}
CPP

g++ -std=c++17 -O2 pram_lab.cpp -o pram_lab
./pram_lab scan 3 1 4 1 5 9 2 6 | python3 -m json.tool > /dev/null
echo "Compiled pram_lab successfully"


Compiled pram_lab successfully


In [9]:
import json, math, subprocess, sys, importlib.util, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, FancyArrowPatch
from IPython.display import display, clear_output, Markdown

if importlib.util.find_spec('ipywidgets') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets'])
import ipywidgets as widgets
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

plt.rcParams['figure.dpi'] = 130

BASE_SEED = [3,1,4,1,5,9,2,6,5,3,5,8,9,7,9,3,2,3,8,4,6,2,6,4,3,3,8,3,2,7,9,5]

def demo_values(n):
    return BASE_SEED[:n]

def run_cpp(mode, values):
    cmd = ['./pram_lab', mode] + [str(v) for v in values]
    out = subprocess.check_output(cmd, text=True)
    return json.loads(out)

def run_legality(scenario, p):
    out = subprocess.check_output(['./pram_lab', 'legality', scenario, str(p)], text=True)
    return json.loads(out)

# quick sanity checks
values = demo_values(8)
scan = run_cpp('scan', values)
assert scan['result'] == np.cumsum(values).tolist()
leg = run_legality('scan-block', 8)
assert leg['CREW'] and not leg['EREW']
print('Sanity checks passed.')


Sanity checks passed.



## 1. Why this prefix-sum algorithm is different

In the algorithm below, each round has the same structure:

- partition the array into blocks of size `2, 4, 8, ...`;
- in each block, take the **last prefix of the left half**;
- all processors in the right half **concurrently read** that value and add it to their local prefix.

So each round performs exactly **n/2 additions**, and the total work is **Θ(n log n)**.



## 2. Reduction demo (fixed redraw)

Use the slider or the **Prev / Next / Play** controls. The current round is highlighted in **red**, already-computed nodes are **teal**, and future nodes are **light gray**.


In [10]:
def draw_reduction(values, round_id):
    trace = run_cpp('reduction', values)
    states = trace['states']
    max_round = len(states) - 1
    round_id = max(0, min(round_id, max_round))
    n = len(states[0]['values'])

    fig = plt.figure(figsize=(12, 5.2))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.45, 1.0])
    ax = fig.add_subplot(gs[0, 0])
    tx = fig.add_subplot(gs[0, 1])
    tx.axis('off')

    coords = []
    for level, st in enumerate(states):
        m = len(st['values'])
        step = 2**level
        xs = [(step - 1)/2 + k*step for k in range(m)]
        y = max_round - level
        coords.append([(x, y) for x in xs])

    # edges
    for level in range(1, len(states)):
        edge_color = '#d1d5db' if level > round_id else ('#e11d48' if level == round_id else '#94a3b8')
        edge_width = 2.6 if level == round_id else 1.5
        for i, (x, y) in enumerate(coords[level]):
            x1, y1 = coords[level-1][2*i]
            x2, y2 = coords[level-1][2*i+1]
            ax.plot([x, x1], [y, y1], color=edge_color, lw=edge_width)
            ax.plot([x, x2], [y, y2], color=edge_color, lw=edge_width)

    # nodes
    for level, st in enumerate(states):
        for i, (x, y) in enumerate(coords[level]):
            if level < round_id:
                face = '#14b8a6'; txtc = 'white'
            elif level == round_id:
                face = '#ef4444'; txtc = 'white'
            else:
                face = '#e5e7eb'; txtc = '#334155'
            circ = Circle((x, y), 0.23, facecolor=face, edgecolor='white', lw=1.5)
            ax.add_patch(circ)
            label = st['values'][i] if level <= round_id else '?'
            ax.text(x, y, str(label), ha='center', va='center', fontsize=10, color=txtc, weight='bold')

    ax.set_title(f'Reduction tree — round {round_id} / {max_round}', fontsize=14, weight='bold')
    ax.set_xlim(-0.7, n-0.3)
    ax.set_ylim(-0.6, max_round + 0.8)
    ax.axis('off')
    ax.text(0, max_round+0.45, 'leaves', fontsize=10, color='#64748b')
    ax.text(n-1.2, 0.25, 'root', fontsize=10, color='#64748b')

    if round_id == 0:
        lines = ['Round 0 = input values.']
    else:
        ops = states[round_id]['ops']
        lines = [f'P{o["p"]}: {o["before_left"]} + {o["before_right"]} → {o["result"]}' for o in ops]
    tx.text(0.02, 0.96, 'Operations in the selected round', fontsize=13, weight='bold', va='top')
    tx.text(0.02, 0.88, '\n'.join(lines), family='monospace', fontsize=11, va='top')

    work_so_far = sum(len(states[r]['ops']) for r in range(1, round_id+1))
    tx.text(0.02, 0.15, f'Time so far: {round_id} rounds\nWork so far: {work_so_far} additions', fontsize=11, color='#334155')
    plt.show()


def make_reduction_widget(n0=8):
    n_dropdown = widgets.Dropdown(options=[8,16,32], value=n0, description='n')
    round_slider = widgets.IntSlider(description='round', min=0, max=int(math.log2(n0)),
                                     value=0, continuous_update=False)
    play = widgets.Play(interval=900, min=0, max=round_slider.max, step=1, value=0)
    # Kernel-side link: Play changes reach Python observers and force a redraw in Colab.
    play_link = widgets.link((play, 'value'), (round_slider, 'value'))
    prev_btn = widgets.Button(description='◀ Prev')
    next_btn = widgets.Button(description='Next ▶')
    out = widgets.Output()

    def redraw(*_):
        values = demo_values(n_dropdown.value)
        with out:
            clear_output(wait=True)
            draw_reduction(values, round_slider.value)

    def sync_n(change):
        max_round = int(math.log2(change['new']))
        round_slider.max = max_round
        play.max = max_round
        round_slider.value = 0
        redraw()

    prev_btn.on_click(lambda _: setattr(round_slider, 'value', max(0, round_slider.value - 1)))
    next_btn.on_click(lambda _: setattr(round_slider, 'value', min(round_slider.max, round_slider.value + 1)))
    n_dropdown.observe(sync_n, names='value')
    round_slider.observe(redraw, names='value')
    redraw()

    display(widgets.VBox([
        widgets.HTML('<b>Reduction execution</b> — red = current round, teal = completed, gray = future'),
        widgets.HBox([n_dropdown, prev_btn, play, next_btn, round_slider]),
        out
    ]))

make_reduction_widget(8)


## 3. Prefix sum on CREW PRAM — interactive execution dashboard

The algorithm below is the **block-doubling / boundary-broadcast** prefix sum from the lecture.

For every round:

1. split the array into blocks of size `2, 4, 8, ...`;
2. in each block, identify the **pivot** = the last prefix of the left half;
3. every processor in the right half reads that same pivot using **Concurrent Read**;
4. each of those processors writes only its own cell, so writes remain **Exclusive**.

Hence the algorithm is naturally **CREW**.

### What to do with the dashboard

- Move **round** to execute one PRAM level at a time.
- Move **processor Pᵢ** to inspect exactly what one processor does in that round.
- Move **target i** to track what range of the original input is already represented by one prefix cell.
- Turn **future edges** on/off to simplify the dependency graph.
- Use **Play** to animate the entire algorithm.

The right panel makes the cost model explicit: **n/2 additions every round**, so total work is `(n/2) log₂ n = Θ(n log n)`.


In [11]:

def contributor_range(target, round_id):
    """Which original cells contribute to target after round_id?"""
    if round_id == 0:
        return target, target
    block = 2 ** round_id
    start = (target // block) * block
    return start, target


def draw_array_boxes(ax, values, highlight=None, title='', selected=None):
    n = len(values)
    ax.set_xlim(-0.55, n - 0.45)
    ax.set_ylim(-0.02, 1.08)
    ax.axis('off')
    for i, v in enumerate(values):
        in_range = highlight is not None and highlight[0] <= i <= highlight[1]
        fc = '#dcfce7' if in_range else '#ffffff'
        ec = '#16a34a' if in_range else '#cbd5e1'
        lw = 2.8 if selected == i else 1.4
        if selected == i:
            ec = '#7c3aed'
        ax.add_patch(Rectangle((i - 0.43, 0.25), 0.86, 0.52,
                               facecolor=fc, edgecolor=ec, lw=lw))
        ax.text(i, 0.51, str(v), ha='center', va='center', fontsize=10,
                weight='bold', color='#0f172a')
        ax.text(i, 0.08, f'{i}', ha='center', va='center', fontsize=8, color='#64748b')
    ax.text(-0.50, 0.98, title, fontsize=11, weight='bold', color='#0f172a')


def active_op_for_processor(state, processor):
    for op in state['ops']:
        if op['p'] == processor:
            return op
    return None


def scan_round_summary(trace, round_id):
    state = trace['states'][round_id]
    n = len(trace['result'])
    total_rounds = len(trace['states']) - 1
    return {
        'round': round_id,
        'block size': state['block_size'],
        'half size': state['half'],
        'active add processors': state['additions'],
        'additions this round': state['additions'],
        'cumulative additions': sum(s['additions'] for s in trace['states'][:round_id+1]),
        'remaining rounds': total_rounds - round_id,
        'total additions': trace['total_additions'],
        'expected total': (n // 2) * int(math.log2(n)),
    }


def draw_scan_dashboard(values, round_id, target, processor, show_future=True):
    trace = run_cpp('scan', values)
    states = trace['states']
    max_round = len(states) - 1
    n = len(values)
    round_id = max(0, min(int(round_id), max_round))
    target = max(0, min(int(target), n - 1))
    processor = max(0, min(int(processor), n - 1))
    state = states[round_id]
    selected_op = active_op_for_processor(state, processor) if round_id > 0 else None

    fig = plt.figure(figsize=(15, 9.5))
    gs = fig.add_gridspec(3, 2,
                          width_ratios=[3.5, 1.35],
                          height_ratios=[0.85, 0.85, 3.4],
                          hspace=.34, wspace=.12)
    ax_input = fig.add_subplot(gs[0, 0])
    ax_state = fig.add_subplot(gs[1, 0])
    ax_graph = fig.add_subplot(gs[2, 0])
    ax_info = fig.add_subplot(gs[:, 1])
    ax_info.axis('off')

    # ---- top: semantic meaning of selected target ----
    hi = contributor_range(target, round_id)
    draw_array_boxes(
        ax_input, values, hi,
        title=f'Original inputs already represented by prefix cell {target}',
        selected=target
    )
    draw_array_boxes(
        ax_state, state['values'], (target, target),
        title=f'Shared-memory state after round {round_id}',
        selected=target
    )

    # ---- dependency graph ----
    ax_graph.set_xlim(-0.55, n - 0.45)
    ax_graph.set_ylim(-0.42, max_round + 0.75)
    ax_graph.axis('off')
    ax_graph.set_title(
        f'CREW block-doubling prefix sum — round {round_id}/{max_round}',
        fontsize=15, weight='bold', loc='left'
    )

    # vertical cell timelines
    for i in range(n):
        ax_graph.plot([i, i], [0, max_round], color='#cbd5e1', lw=2.0, zorder=0)
        ax_graph.text(i, -0.17, f'{i}', ha='center', va='top', fontsize=8, color='#64748b')

    # round/block shading around current transition
    if round_id > 0:
        block_size = state['block_size']
        for b, start in enumerate(range(0, n, block_size)):
            x0 = start - 0.44
            width = min(block_size, n-start) - 0.12
            fc = '#fff7ed' if b % 2 == 0 else '#fef2f2'
            ax_graph.add_patch(Rectangle(
                (x0, round_id-1+0.08), width, 0.84,
                facecolor=fc, edgecolor='none', alpha=.60, zorder=-1
            ))

    # nodes
    for r in range(max_round + 1):
        for i in range(n):
            if r < round_id:
                face, edge = '#dbeafe', '#60a5fa'
            elif r == round_id:
                face, edge = '#fee2e2', '#ef4444'
            else:
                face, edge = '#f8fafc', '#d1d5db'
            lw = 2.8 if (r == round_id and i == processor) else 1.0
            if r == round_id and i == processor:
                edge = '#7c3aed'
            ax_graph.add_patch(Circle((i, r), 0.085, facecolor=face, edgecolor=edge, lw=lw, zorder=3))

    # dependency arrows: past / current / optional future
    for r in range(1, max_round + 1):
        st = states[r]
        if r > round_id and not show_future:
            continue

        by_source = {}
        for op in st['ops']:
            by_source.setdefault(op['left'], []).append(op)

        if r < round_id:
            base_color, base_lw, alpha = '#60a5fa', 1.55, 1.0
        elif r == round_id:
            base_color, base_lw, alpha = '#dc2626', 2.35, 1.0
        else:
            base_color, base_lw, alpha = '#d1d5db', 1.0, .78

        for src, ops in by_source.items():
            # CR visual marker for current round whenever fan-out > 1
            if r == round_id and len(ops) > 1:
                ax_graph.add_patch(Circle((src, r-1), 0.19, fill=False,
                                          edgecolor='#22c55e', lw=2.4, zorder=5))
                ax_graph.text(src-0.17, r-1-0.25, 'CR', color='#16a34a',
                              fontsize=10, weight='bold', zorder=6)

            for op in ops:
                is_selected = (r == round_id and op['p'] == processor)
                color = '#7c3aed' if is_selected else base_color
                lw = 3.8 if is_selected else base_lw
                arr = FancyArrowPatch(
                    (src, r-1), (op['right'], r),
                    arrowstyle='->', mutation_scale=12,
                    lw=lw, color=color, alpha=alpha,
                    connectionstyle='arc3,rad=0.0', zorder=4
                )
                ax_graph.add_patch(arr)

    # labels for round axis
    for r in range(max_round + 1):
        label = 'input' if r == 0 else f'R{r}'
        ax_graph.text(-0.50, r, label, ha='right', va='center', fontsize=9, color='#64748b')

    # target vertical trace up to selected round
    ax_graph.plot([target]*(round_id+1), list(range(round_id+1)),
                  color='#0f766e', lw=2.0, alpha=.70, zorder=2)
    ax_graph.text(target+0.12, round_id+0.10, f'target {target}',
                  color='#0f766e', fontsize=9, weight='bold')

    # ---- info panel ----
    ax_info.text(.02, .97, 'ACTION PANEL', fontsize=14, weight='bold', va='top', color='#0f172a')

    # 1. processor action
    ax_info.text(.02, .90, f'Processor P{processor}', fontsize=12.5, weight='bold', color='#7c3aed')
    if round_id == 0:
        action_text = 'No computation yet: input state.'
    elif selected_op is None:
        action_text = (
            f'P{processor} is idle in this round.\n'
            f'Its prefix value remains {state["values"][processor]}.'
        )
    else:
        action_text = (
            f'READ  X[{selected_op["right"]}] = {selected_op["before_right"]}\n'
            f'CR    X[{selected_op["left"]}] = {selected_op["before_left"]}\n'
            f'ADD   {selected_op["before_right"]} + {selected_op["before_left"]}\n'
            f'WRITE X[{selected_op["right"]}] = {selected_op["result"]}'
        )
    ax_info.text(.04, .855, action_text, fontsize=10.6, family='monospace', va='top', color='#334155')

    # 2. current target semantics
    ax_info.text(.02, .64, f'Target prefix cell {target}', fontsize=12.5, weight='bold', color='#0f766e')
    target_text = (
        f'Current value: {state["values"][target]}\n'
        f'Represents input indices {hi[0]}..{hi[1]}\n'
        f'Final S[{target}] = {trace["result"][target]}'
    )
    ax_info.text(.04, .595, target_text, fontsize=10.6, family='monospace', va='top', color='#334155')

    # 3. cost model
    metrics = scan_round_summary(trace, round_id)
    ax_info.text(.02, .44, 'Cost model', fontsize=12.5, weight='bold', color='#0f172a')
    cost_text = (
        f'block size           {metrics["block size"]}\n'
        f'active add processors {metrics["active add processors"]}\n'
        f'additions this round  {metrics["additions this round"]}\n'
        f'cumulative additions  {metrics["cumulative additions"]}\n'
        f'total additions       {metrics["total additions"]}\n'
        f'rounds total          {max_round}'
    )
    ax_info.text(.04, .397, cost_text, fontsize=10.4, family='monospace', va='top', color='#334155')

    # constant-work callout
    if round_id > 0:
        ax_info.add_patch(Rectangle((.02, .09), .94, .13, transform=ax_info.transAxes,
                                    facecolor='#ecfdf5', edgecolor='#86efac', lw=1.2))
        ax_info.text(.49, .155,
                     f'Round work = n/2 = {n//2} additions\nfor every nonzero round',
                     ha='center', va='center', fontsize=11, weight='bold',
                     color='#166534', transform=ax_info.transAxes)
    else:
        ax_info.add_patch(Rectangle((.02, .09), .94, .13, transform=ax_info.transAxes,
                                    facecolor='#eff6ff', edgecolor='#93c5fd', lw=1.2))
        ax_info.text(.49, .155,
                     f'{max_round} rounds ahead\nEach will execute {n//2} additions',
                     ha='center', va='center', fontsize=11, weight='bold',
                     color='#1d4ed8', transform=ax_info.transAxes)

    plt.show()

    # concrete operations for current round as a table
    if round_id > 0:
        rows = []
        for op in state['ops']:
            rows.append({
                'processor': f'P{op["p"]}',
                'local read': f'X[{op["right"]}]={op["before_right"]}',
                'concurrent read': f'X[{op["left"]}]={op["before_left"]}',
                'write': f'X[{op["right"]}]={op["result"]}',
            })
        df = pd.DataFrame(rows)
        display(Markdown(f'**All active processors in round {round_id}** — exactly `{len(df)} = n/2` additions'))
        display(df)


def make_scan_dashboard(n0=8):
    n_dropdown = widgets.Dropdown(options=[8,16,32], value=n0, description='n')
    target_slider = widgets.IntSlider(description='target i', min=0, max=n0-1,
                                      value=n0-1, continuous_update=False)
    processor_slider = widgets.IntSlider(description='processor Pᵢ', min=0, max=n0-1,
                                         value=n0-1, continuous_update=False)

    trace0 = run_cpp('scan', demo_values(n0))
    max_round0 = len(trace0['states']) - 1
    round_slider = widgets.IntSlider(description='round', min=0, max=max_round0,
                                     value=0, continuous_update=False)
    play = widgets.Play(interval=1200, min=0, max=max_round0, step=1, value=0)
    play_link = widgets.link((play, 'value'), (round_slider, 'value'))
    prev_btn = widgets.Button(description='◀ Prev')
    next_btn = widgets.Button(description='Next ▶')
    future_cb = widgets.Checkbox(value=True, description='future edges')
    out = widgets.Output()

    def redraw(*_):
        with out:
            clear_output(wait=True)
            draw_scan_dashboard(
                demo_values(n_dropdown.value),
                round_slider.value,
                target_slider.value,
                processor_slider.value,
                future_cb.value,
            )

    def sync_n(change):
        n = change['new']
        tr = run_cpp('scan', demo_values(n))
        max_round = len(tr['states']) - 1
        round_slider.max = max_round
        play.max = max_round
        round_slider.value = 0
        target_slider.max = n-1
        processor_slider.max = n-1
        target_slider.value = n-1
        processor_slider.value = n-1
        redraw()

    prev_btn.on_click(lambda _: setattr(round_slider, 'value', max(0, round_slider.value-1)))
    next_btn.on_click(lambda _: setattr(round_slider, 'value', min(round_slider.max, round_slider.value+1)))

    n_dropdown.observe(sync_n, names='value')
    round_slider.observe(redraw, names='value')
    target_slider.observe(redraw, names='value')
    processor_slider.observe(redraw, names='value')
    future_cb.observe(redraw, names='value')

    redraw()
    display(widgets.VBox([
        widgets.HTML('<b>CREW prefix-sum execution dashboard</b>'),
        widgets.HBox([n_dropdown, target_slider, processor_slider]),
        widgets.HBox([prev_btn, play, next_btn, round_slider, future_cb]),
        out
    ]))

make_scan_dashboard(8)


## 4. Verification and cost checks

This cell checks the examples rather than just drawing them.

For `n = 8, 16, 32` it verifies that:

- the C++ result equals `numpy.cumsum`;
- the number of rounds is `log₂ n`;
- **every nonzero round executes exactly `n/2` additions**;
- total work is `(n/2) log₂ n`;
- the representative access pattern requires **CREW** (not EREW) once a pivot fans out to several readers.


In [12]:

checks = []
for n in [8,16,32]:
    values = demo_values(n)
    tr = run_cpp('scan', values)
    rounds = len(tr['states']) - 1
    additions = [s['additions'] for s in tr['states'][1:]]
    legal = run_legality('scan-block', n)
    checks.append({
        'n': n,
        'correct prefix sum': tr['result'] == np.cumsum(values).tolist(),
        'rounds': rounds,
        'expected rounds': int(math.log2(n)),
        'additions / round': additions,
        'constant n/2?': all(x == n//2 for x in additions),
        'total additions': tr['total_additions'],
        'expected total': (n//2) * int(math.log2(n)),
        'CREW legal': legal['CREW'],
        'EREW legal': legal['EREW'],
    })

check_df = pd.DataFrame(checks)
display(check_df)
assert check_df['correct prefix sum'].all()
assert check_df['constant n/2?'].all()
assert (check_df['rounds'] == check_df['expected rounds']).all()
assert (check_df['total additions'] == check_df['expected total']).all()
assert check_df['CREW legal'].all()
print('All algorithmic checks passed.')


,n,correct prefix sum,rounds,expected rounds,additions / round,constant n/2?,total additions,expected total,CREW legal,EREW legal
0,8,True,3,3,"[4, 4, 4]",True,12,12,True,False
1,16,True,4,4,"[8, 8, 8, 8]",True,32,32,True,False
2,32,True,5,5,"[16, 16, 16, 16, 16]",True,80,80,True,False


All algorithmic checks passed.


## 5. Why the work is Θ(n log n): compare with reduction

This plot is useful in class because it makes the difference visible immediately:

- a **reduction** activates fewer processors at every level;
- this **prefix sum** keeps exactly `n/2` additions active at every level.

Try changing `n` in the cell below.


In [13]:

def plot_round_work(n=32):
    values = demo_values(n)
    red = run_cpp('reduction', values)
    scan = run_cpp('scan', values)

    red_work = [len(s['ops']) for s in red['states'][1:]]
    scan_work = [s['additions'] for s in scan['states'][1:]]
    rounds = np.arange(1, len(scan_work)+1)

    fig, ax = plt.subplots(figsize=(9,4.5))
    ax.plot(rounds, red_work, marker='o', linewidth=2, label='Reduction')
    ax.plot(rounds, scan_work, marker='o', linewidth=2, label='CREW prefix sum')
    ax.set_xticks(rounds)
    ax.set_xlabel('PRAM round')
    ax.set_ylabel('additions in the round')
    ax.set_title(f'Work per round for n={n}')
    ax.grid(alpha=.25)
    ax.legend()
    plt.show()

    display(pd.DataFrame({
        'round': rounds,
        'reduction additions': red_work,
        'prefix-sum additions': scan_work,
    }))
    print(f'Reduction total additions: {sum(red_work)} = n-1')
    print(f'Prefix-sum total additions: {sum(scan_work)} = (n/2) log2(n)')

n_work = widgets.ToggleButtons(options=[8,16,32], value=32, description='n')
widgets.interact(plot_round_work, n=n_work)


interactive(children=(ToggleButtons(description='n', index=2, options=(8, 16, 32), value=32), Output()), _dom_…

<function __main__.plot_round_work(n=32)>

## 6. Takeaway

For the CREW block-doubling prefix sum:

$T_p(n)=\Theta(\log n), \qquad W(n)=\Theta(n\log n).$

The reason is now visible in the dashboard:

- the **number of rounds** is logarithmic because blocks double every round;
- the **number of additions per round is constant at `n/2`**;
- the shared pivot of each block is read concurrently, which is precisely where **CREW** enters the algorithm.

A useful exercise is to select one processor and follow it across the rounds: in some rounds it is idle, and in others it receives a broadcast prefix from the left half of its block.
